In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.13f' % x)
pd.set_option('display.max_columns', None)

from scipy.stats import wilcoxon

In [2]:
df = pd.read_excel('all-model-ablation.xlsx', index_col=0)
df.head()

,w/o H1,w/o H2,w/o H3,w/o H4 (full),w/o H4 (avg),w/o H4 (max),w/o H4 (last),proposed,Mamba,w/ H2,w/o H2 (more exp)
EC,33.0798479087452,32.3193916349810,39.5437262357414,34.6007604562738,32.6996197718631,41.8250950570342,33.0798479087452,42.5855513307985,30.4182509505703,34.2205323193916,33.4600760456274
FD,69.2962542565267,68.5868331441544,69.2111237230420,70.0056753688990,70.0340522133939,69.0124858115777,66.8274687854711,69.2962542565267,68.7854710556186,69.1543700340522,69.4948921679909
HW,60.2352941176471,54.0000000000000,58.0000000000000,47.2941176470588,59.8823529411765,45.5294117647059,56.7058823529412,60.8235294117647,30.9411764705882,60.4705882352941,56.0000000000000
HB,80.9756097560976,79.0243902439024,80.0000000000000,80.0000000000000,80.4878048780488,80.9756097560976,78.5365853658537,80.4878048780488,77.0731707317073,80.9756097560976,80.9756097560976
JV,98.6486486486486,98.1081081081081,98.6486486486486,98.6486486486486,98.1081081081081,98.9189189189189,98.9189189189189,98.6486486486486,97.2972972972973,97.8378378378378,98.9189189189189


In [3]:
df_norm = ((df.T - df.mean(axis=1)) / df.std(axis=1)).T
wilcoxon_res = list()
for c in df.columns:
    if c == 'proposed':
        continue
    res = wilcoxon((df_norm['proposed'] - df_norm[c]), alternative='greater')
    wilcoxon_res.append((c, res.pvalue))

df_wilcoxon = pd.DataFrame(wilcoxon_res, columns=['Classifier', 'p-value']).set_index('Classifier')
df_wilcoxon.T

Classifier,w/o H1,w/o H2,w/o H3,w/o H4 (full),w/o H4 (avg),w/o H4 (max),w/o H4 (last),Mamba,w/ H2,w/o H2 (more exp)
p-value,0.2173780858792,0.0000021142044,0.0712660396577,0.0033155411482,0.0441488386581,0.0039761469015,0.0005499539844,0.0000065906045,0.0105441486636,0.1057776934253


In [4]:
df_rank = df.rank(axis=1, method='min',ascending=False).astype(int)
df_rank.head()

,w/o H1,w/o H2,w/o H3,w/o H4 (full),w/o H4 (avg),w/o H4 (max),w/o H4 (last),proposed,Mamba,w/ H2,w/o H2 (more exp)
EC,7,10,3,4,9,2,7,1,11,5,6
FD,4,10,6,2,1,8,11,5,9,7,3
HW,3,8,5,9,4,10,6,1,11,2,7
HB,2,9,7,7,5,2,10,5,11,2,1
JV,4,8,4,4,8,1,1,4,11,10,3


In [5]:
labels = df.columns
scores = df.to_numpy()
ranks = df_rank.to_numpy()

n_datasets, n_estimators = df.shape

avg_ranks = ranks.mean(axis=0)
ordered_labels_ranks = np.array(
    [(l, float(r)) for r, l in sorted(zip(avg_ranks, labels))], dtype=object
)
ordered_labels = np.array([la for la, _ in ordered_labels_ranks], dtype=str)
ordered_avg_ranks = np.array([r for _, r in ordered_labels_ranks], dtype=np.float32)
ordered_ranks = ranks[:, [np.where(np.array(labels) == r)[0][0] for r in ordered_labels]]

print(ordered_labels_ranks)

[['proposed' 3.2333333333333334]
 ['w/o H1' 3.566666666666667]
 ['w/o H2 (more exp)' 4.166666666666667]
 ['w/o H3' 4.533333333333333]
 ['w/o H4 (avg)' 4.566666666666666]
 ['w/o H4 (max)' 5.066666666666666]
 ['w/ H2' 5.166666666666667]
 ['w/o H4 (full)' 6.133333333333334]
 ['w/o H4 (last)' 6.333333333333333]
 ['w/o H2' 7.6]
 ['Mamba' 9.1]]
